# 06 — EXP_11 + EXP_12: Passage-level SHAP and LIME ↔ SHAP Agreement

## Why this notebook combines two experiments

EXP_10 Stage B already produced 205 questions × 16 random subset samples + per-mask LLM predictions ([results/exp_10_lime_passage/stage_b_retrievalchanged_mhop.jsonl](../results/exp_10_lime_passage/stage_b_retrievalchanged_mhop.jsonl)). Computing SHAP and LIME-SHAP agreement on top of that data requires **zero new Groq calls**:

- **EXP_11 KernelSHAP** uses the same `(mask, prediction)` pairs as EXP_10 but with the SHAP kernel weighting (`w(S) = (k−1) / [C(k, |S|) · |S| · (k − |S|)]`) instead of LIME's uniform ridge. It's a weighted regression on the same data.
- **EXP_12 agreement** is a one-shot per-question comparison: top-1 / top-3 overlap + Spearman rank correlation between LIME and SHAP attributions.

**Cost**: $0 (all CPU; no Groq calls). Wall time: ~30 sec total.

## No-RAG anchor for SHAP

EXP_11 optionally adds a synthetic all-zeros sample per question — its prediction is the No-RAG result from `exp_01_base_llm__test_1273/predictions.jsonl`. With a high constraint weight (1e6), this forces KernelSHAP's intercept to anchor at the No-RAG endpoint, giving cleaner Shapley estimates. Without it, SHAP only sees the all-ones anchor (the full prediction).

## 1. Setup

In [1]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

from src.xai.shap_passage import run_shap_from_lime_batch
from src.xai.agreement import run_agreement_batch

print("Repo root:", REPO_ROOT)

LIME_PATH = REPO_ROOT / "results/exp_10_lime_passage/stage_b_retrievalchanged_mhop.jsonl"
SHAP_PATH = REPO_ROOT / "results/exp_11_shap_passage/stage_b_retrievalchanged_mhop.jsonl"
AGR_PATH  = REPO_ROOT / "results/exp_12_agreement/stage_b_retrievalchanged_mhop.jsonl"

assert LIME_PATH.exists(), f"missing {LIME_PATH} — run EXP_10 Stage B first."
print(f"LIME input: {LIME_PATH.name} ({LIME_PATH.stat().st_size / 1024:.1f} KB)")

Repo root: /Users/rajak/Workstation/Projects/myGitHub/thesis-project
LIME input: stage_b_retrievalchanged_mhop.jsonl (994.3 KB)


## 2. Build the No-RAG anchor map from EXP_01

In [2]:
e1 = pd.DataFrame([
    json.loads(l) for l in (REPO_ROOT / "results/exp_01_base_llm__test_1273/predictions.jsonl").read_text().splitlines()
]).set_index("question_id")
no_rag_pred_map = dict(zip(e1.index, e1["pred_letter"]))
print(f"No-RAG anchor map: {len(no_rag_pred_map)} entries")

No-RAG anchor map: 1273 entries


## 3. EXP_11 — Run KernelSHAP on the EXP_10 Stage B data

No new Groq calls. Reads each LIME record, applies SHAP kernel weights to the same `(mask, prediction)` samples, fits weighted regression, writes per-question Shapley values.

In [3]:
summary = run_shap_from_lime_batch(
    lime_jsonl=LIME_PATH,
    output_path=SHAP_PATH,
    no_rag_pred_map=no_rag_pred_map,
)
print(json.dumps(summary, indent=2))

[shap] resuming — 205 (qid, arch) rows already done


KernelSHAP:   0%|          | 0/205 [00:00<?, ?it/s]

{
  "output_path": "/Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_11_shap_passage/stage_b_retrievalchanged_mhop.jsonl",
  "method": "kernel_shap",
  "n_rows_written_this_run": 0,
  "n_rows_already_done": 205,
  "n_with_no_rag_anchor": 0,
  "wall_time_s_this_run": 0.019215106964111328
}


## 4. EXP_11 — SHAP signal density & attribution distribution

In [4]:
shap_rows = [json.loads(l) for l in SHAP_PATH.read_text().splitlines()]
print(f"=== EXP_11 KernelSHAP ({len(shap_rows)} questions) ===")
n_corr = sum(1 for r in shap_rows if r["correctness_score_variance"] > 0)
n_same = sum(1 for r in shap_rows if r["sameletter_score_variance"] > 0)
print(f"  correctness variance > 0: {n_corr}/{len(shap_rows)} = {n_corr/len(shap_rows):.1%}")
print(f"  sameletter   variance > 0: {n_same}/{len(shap_rows)} = {n_same/len(shap_rows):.1%}")
print(f"  with No-RAG anchor       : {sum(1 for r in shap_rows if r['no_rag_anchor_used'])}/{len(shap_rows)}")

# Top-1 magnitude distribution (on signal questions)
max_corr = [max(abs(p['correctness_shap']) for p in r['passages']) for r in shap_rows if r['correctness_score_variance'] > 0]
if max_corr:
    print(f"\n  top-1 |correctness_shap| on signal Q (n={len(max_corr)}):")
    print(f"    mean: {np.mean(max_corr):.3f}  median: {np.median(max_corr):.3f}  range: [{min(max_corr):.3f}, {max(max_corr):.3f}]")

=== EXP_11 KernelSHAP (205 questions) ===
  correctness variance > 0: 185/205 = 90.2%
  sameletter   variance > 0: 205/205 = 100.0%
  with No-RAG anchor       : 205/205

  top-1 |correctness_shap| on signal Q (n=185):
    mean: 1.025  median: 0.912  range: [0.126, 3.893]


## 5. EXP_12 — LIME ↔ SHAP agreement

Per-question top-1 / top-3 overlap and Spearman rank correlation.

In [5]:
agr_summary = run_agreement_batch(
    lime_jsonl=LIME_PATH,
    shap_jsonl=SHAP_PATH,
    output_path=AGR_PATH,
)
print(json.dumps(agr_summary, indent=2))

[agreement] LIME rows=205, SHAP rows=205, common=205


agreement:   0%|          | 0/205 [00:00<?, ?it/s]

{
  "output_path": "/Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_12_agreement/stage_b_retrievalchanged_mhop.jsonl",
  "method": "lime_shap_agreement",
  "n_rows_written": 205,
  "n_rows_skipped": 0,
  "n_lime_rows": 205,
  "n_shap_rows": 205,
  "n_common": 205,
  "wall_time_s": 0.08812999725341797
}


## 6. Aggregate agreement statistics

In [6]:
agr_rows = [json.loads(l) for l in AGR_PATH.read_text().splitlines()]
df = pd.DataFrame(agr_rows)
print(f"=== EXP_12 agreement ({len(df)} questions) ===")

for signal in ("correctness", "sameletter"):
    print(f"\n--- {signal} signal ---")
    top1 = df[f"{signal}_top1"].dropna()
    top3 = df[f"{signal}_top3_overlap"].dropna()
    rho  = df[f"{signal}_spearman"].dropna()
    print(f"  top1 agreement   : {top1.mean():.3f}  (n={len(top1)} questions; NaN excluded)")
    print(f"  top3 overlap     : mean {top3.mean():.3f}  median {top3.median():.3f}  (n={len(top3)})")
    print(f"  Spearman ρ       : mean {rho.mean():.3f}  median {rho.median():.3f}  (n={len(rho)})")
    # Distribution buckets
    n_strong = (rho > 0.7).sum()
    n_moderate = ((rho > 0.3) & (rho <= 0.7)).sum()
    n_weak = ((rho > -0.3) & (rho <= 0.3)).sum()
    n_anti = (rho <= -0.3).sum()
    print(f"  ρ distribution   : strong (>0.7)={n_strong}, moderate (0.3-0.7)={n_moderate}, weak (-0.3-0.3)={n_weak}, anti (<-0.3)={n_anti}")

=== EXP_12 agreement (205 questions) ===

--- correctness signal ---
  top1 agreement   : 0.515  (n=134 questions; NaN excluded)
  top3 overlap     : mean 0.556  median 0.667  (n=205)
  Spearman ρ       : mean 0.632  median 0.706  (n=134)
  ρ distribution   : strong (>0.7)=68, moderate (0.3-0.7)=46, weak (-0.3-0.3)=18, anti (<-0.3)=2

--- sameletter signal ---
  top1 agreement   : 0.472  (n=161 questions; NaN excluded)
  top3 overlap     : mean 0.504  median 0.667  (n=205)
  Spearman ρ       : mean 0.653  median 0.734  (n=161)
  ρ distribution   : strong (>0.7)=87, moderate (0.3-0.7)=53, weak (-0.3-0.3)=20, anti (<-0.3)=1


## 7. Stratify by change-type (fix / break / both_wrong)

In [7]:
e5 = pd.DataFrame([
    json.loads(l) for l in (REPO_ROOT / "results/exp_05_multi_hop_rag__test_1273/predictions.jsonl").read_text().splitlines()
]).set_index("question_id")

def _classify(qid: str) -> str:
    if qid not in e1.index or qid not in e5.index:
        return "unknown"
    nr = bool(e1.loc[qid, "is_correct"])
    mh = bool(e5.loc[qid, "is_correct"])
    if not nr and mh: return "fix"
    if nr and not mh: return "break"
    if not nr and not mh: return "both_wrong"
    return "unknown"

df["change_type"] = df.question_id.map(_classify)
print(f"Change-type distribution: {dict(df.change_type.value_counts())}")

for ct in ("fix", "break", "both_wrong"):
    sub = df[df.change_type == ct]
    if sub.empty: continue
    top1c = sub["correctness_top1"].dropna()
    rho_c = sub["correctness_spearman"].dropna()
    top1s = sub["sameletter_top1"].dropna()
    rho_s = sub["sameletter_spearman"].dropna()
    print(f"\n  {ct:11s} (n={len(sub):>3d}):")
    print(f"    correctness:  top1 agree = {top1c.mean():.3f}  ρ mean = {rho_c.mean():.3f}  (n={len(top1c)})")
    print(f"    sameletter :  top1 agree = {top1s.mean():.3f}  ρ mean = {rho_s.mean():.3f}  (n={len(top1s)})")

Change-type distribution: {'fix': np.int64(101), 'break': np.int64(73), 'both_wrong': np.int64(31)}

  fix         (n=101):
    correctness:  top1 agree = 0.514  ρ mean = 0.605  (n=72)
    sameletter :  top1 agree = 0.514  ρ mean = 0.605  (n=72)

  break       (n= 73):
    correctness:  top1 agree = 0.569  ρ mean = 0.656  (n=51)
    sameletter :  top1 agree = 0.475  ρ mean = 0.732  (n=61)

  both_wrong  (n= 31):
    correctness:  top1 agree = 0.273  ρ mean = 0.707  (n=11)
    sameletter :  top1 agree = 0.357  ρ mean = 0.602  (n=28)


## 8. Sanity — pick 3 questions and show LIME vs SHAP top chunks side-by-side

In [8]:
lime_rows = {(r["question_id"], r["architecture"]): r for r in [json.loads(l) for l in LIME_PATH.read_text().splitlines()]}
shap_map  = {(r["question_id"], r["architecture"]): r for r in shap_rows}

# Pick first 3 questions with signal
shown = 0
for r in shap_rows:
    if r["correctness_score_variance"] == 0:
        continue
    key = (r["question_id"], r["architecture"])
    lime_r = lime_rows[key]
    print(f"\n=== {r['question_id']}  gold={r['gold_letter']}  full_pred={r['full_pred_letter']} ===")
    print(f"  LIME top-3 by |correctness_coef|:")
    lime_top = sorted(lime_r["passages"], key=lambda p: -abs(p["correctness_coef"]))[:3]
    for p in lime_top:
        print(f"    rank={p['rank']:>2d}  {p['chunk_id'][:38]:<38}  coef={p['correctness_coef']:+.4f}")
    print(f"  SHAP top-3 by |correctness_shap|:")
    shap_top = sorted(r["passages"], key=lambda p: -abs(p["correctness_shap"]))[:3]
    for p in shap_top:
        print(f"    rank={p['rank']:>2d}  {p['chunk_id'][:38]:<38}  shap={p['correctness_shap']:+.4f}")
    # Top-1 match?
    top1_match = lime_top[0]["chunk_id"] == shap_top[0]["chunk_id"]
    print(f"  top-1 match: {'✓' if top1_match else '✗'}")
    shown += 1
    if shown >= 3: break


=== medqa_11453  gold=D  full_pred=D ===
  LIME top-3 by |correctness_coef|:
    rank=11  First_Aid_Step1_chunk_00089             coef=+0.4572
    rank= 3  InternalMed_Harrison_chunk_06517        coef=-0.4195
    rank= 8  Surgery_Schwartz_chunk_05487            coef=+0.4149
  SHAP top-3 by |correctness_shap|:
    rank= 8  Surgery_Schwartz_chunk_05487            shap=+0.5805
    rank= 3  InternalMed_Harrison_chunk_06517        shap=-0.3824
    rank=11  First_Aid_Step1_chunk_00089             shap=+0.3720
  top-1 match: ✗

=== medqa_11468  gold=B  full_pred=B ===
  LIME top-3 by |correctness_coef|:
    rank= 8  Surgery_Schwartz_chunk_03425            coef=+0.4801
    rank= 4  Surgery_Schwartz_chunk_03418            coef=+0.4491
    rank=10  Surgery_Schwartz_chunk_03420            coef=+0.3660
  SHAP top-3 by |correctness_shap|:
    rank= 8  Surgery_Schwartz_chunk_03425            shap=+0.5146
    rank= 4  Surgery_Schwartz_chunk_03418            shap=+0.4672
    rank=10  Surgery_Schwartz

---

**Next.** Phase 7 — confidence-aware rejection. The per-question agreement scores written to `results/exp_12_agreement/` feed Phase 7's confidence vector alongside Faithfulness and retrieval scores. No more LLM calls until the rejection-threshold sweep (which itself uses cached predictions).